# 🚗 Driver Behavior Recognition — Kaggle Complete Pipeline
**Repo:** [tunaleeZ/Driver_behaviour](https://github.com/tunaleeZ/Driver_behaviour)  
**Nhận diện 5 hành vi:** `normal`, `drowsy`, `sleepy`, `drinking`, `phone`  

---

### ⚙️ Hướng dẫn cài đặt trên Kaggle:
1. **Accelerator:** Vào menu bên phải (*Notebook options*) -> **Accelerator** -> Chọn **GPU T4 x2** (hoặc GPU P100).
2. **Internet:** Vào menu bên phải -> **Internet** -> Chọn **On** (cần thiết để clone repo từ GitHub và tải file trọng số MediaPipe Face Landmarker).
3. **Dataset:** Đã upload file zip chứa 2 video lên Kaggle và đặt tên dataset là **`driver_behaviour_dataset`**. Khi đó dataset sẽ tự động được mount vào `/kaggle/input/driver-behaviour-dataset`.


In [ ]:
# ==========================================
# 1. KIỂM TRA MÔI TRƯỜNG & GPU
# ==========================================
import sys, os, subprocess
from pathlib import Path
import torch

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM khả dụng: {vram:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào Notebook options -> Accelerator -> Chọn GPU T4 x2.")


## 2. Clone Repository & Cài Đặt Dependencies
Clone toàn bộ mã nguồn repo GitHub vào `/kaggle/working` và cài đặt gói dự án `dbm` cùng các thư viện bổ trợ.


In [ ]:
# ==========================================
# 2. CLONE GITHUB REPO & CÀI ĐẶT
# ==========================================
REPO_URL = "https://github.com/tunaleeZ/Driver_behaviour.git"
WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "Driver_behaviour"

if not REPO_DIR.exists():
    print(f"Đang clone repo từ {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo đã có sẵn tại {REPO_DIR}")

# Chuyển thư mục làm việc về REPO_DIR
os.chdir(REPO_DIR)
print("Thư mục làm việc hiện tại:", os.getcwd())

# Cài đặt các gói phụ thuộc bổ sung (Kaggle đã có sẵn torch, numpy, opencv)
print("Đang cài đặt thư viện cần thiết...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe>=0.10.0", "imageio-ffmpeg>=0.4.9", "openpyxl>=3.1.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"], check=True)

# Thêm src vào sys.path
sys.path.insert(0, str(REPO_DIR / "src"))

import mediapipe as mp
import dbm
print("MediaPipe version:", mp.__version__)
print("Package dbm đã import thành công!")


## 3. Tải Pretrained Model MediaPipe Face Landmarker
Model trọng số `face_landmarker.task` thuộc diện `.gitignore` trên GitHub. Đoạn mã này sẽ tự động tải trực tiếp từ Google Storage chính thức của MediaPipe.


In [ ]:
# ==========================================
# 3. TẢI MODEL FACE LANDMARKER (.task)
# ==========================================
import urllib.request

MODELS_DIR = REPO_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
LANDMARKER_PATH = MODELS_DIR / "face_landmarker.task"
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"

if not LANDMARKER_PATH.exists() or LANDMARKER_PATH.stat().st_size == 0:
    print(f"Đang tải face_landmarker.task từ Google Storage...")
    urllib.request.urlretrieve(MODEL_URL, LANDMARKER_PATH)
    file_size_mb = LANDMARKER_PATH.stat().st_size / 1e6
    print(f"Tải thành công: {LANDMARKER_PATH.name} ({file_size_mb:.2f} MB)")
else:
    file_size_mb = LANDMARKER_PATH.stat().st_size / 1e6
    print(f"Model đã có sẵn: {LANDMARKER_PATH.name} ({file_size_mb:.2f} MB)")

# Đặt biến môi trường để toàn bộ code trích xuất tự động dùng model này
os.environ["DBM_FACE_LANDMARKER_PATH"] = str(LANDMARKER_PATH)


## 4. Quét & Đồng Bộ 2 Video Đã Upload Vào `raw/public/`
Tự động tìm kiếm 2 video trong `/kaggle/input` (hỗ trợ cả trường hợp Kaggle tự giải nén hoặc còn giữ file zip) và đưa vào đúng vị trí `driver_behavior_dataset_v01/raw/public/`.


In [ ]:
# ==========================================
# 4. TÌM VIDEO TỪ KAGGLE DATASET VÀ COPY VÀO RAW
# ==========================================
import shutil, zipfile

KAGGLE_INPUT = Path("/kaggle/input")
RAW_PUBLIC_DIR = REPO_DIR / "driver_behavior_dataset_v01" / "raw" / "public"
RAW_PUBLIC_DIR.mkdir(parents=True, exist_ok=True)

video_extensions = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
found_videos = []

# Tìm kiếm toàn bộ video hoặc file zip trong /kaggle/input
for root, _, files in os.walk(KAGGLE_INPUT):
    for f in files:
        p = Path(root) / f
        if p.suffix.lower() in video_extensions:
            found_videos.append(p)
        elif p.suffix.lower() == ".zip":
            print(f"Phát hiện file zip: {p.name}, đang giải nén...")
            with zipfile.ZipFile(p, "r") as z:
                for member in z.namelist():
                    if Path(member).suffix.lower() in video_extensions:
                        print(f"  -> Giải nén: {member}")
                        z.extract(member, RAW_PUBLIC_DIR)
                        extracted = RAW_PUBLIC_DIR / member
                        if extracted.parent != RAW_PUBLIC_DIR:
                            shutil.move(str(extracted), str(RAW_PUBLIC_DIR / extracted.name))

# Cập nhật danh sách video
found_videos = []
for root, _, files in os.walk(KAGGLE_INPUT):
    for f in files:
        p = Path(root) / f
        if p.suffix.lower() in video_extensions:
            found_videos.append(p)

print(f"\nTìm thấy {len(found_videos)} file video trong /kaggle/input:")
for v in found_videos:
    size_mb = v.stat().st_size / 1e6
    print(f"  - {v.name} ({size_mb:.2f} MB)")
    dst = RAW_PUBLIC_DIR / v.name
    if not dst.exists():
        try:
            os.symlink(v, dst)
        except OSError:
            shutil.copy2(v, dst)

available_videos = list(RAW_PUBLIC_DIR.glob("*.mp4")) + list(RAW_PUBLIC_DIR.glob("*.avi"))
print(f"\nDanh sách video sẵn sàng trong {RAW_PUBLIC_DIR.name}:")
for v in available_videos:
    print(f"  [OK] {v.name}")


## 5. Đồng Bộ Metadata Theo Đúng 2 Video Hiện Có
File `metadata.xlsx` gốc có thông tin của 5 video. Vì bạn tải và upload 2 video, ô code dưới đây sẽ tự động lọc metadata giữ đúng các dòng thuộc về 2 video thực tế có trên đĩa. Nhờ vậy, pipeline sẽ chạy trơn tru 100% mà không bị lỗi `Video file not found`.


In [ ]:
# ==========================================
# 5. LỌC METADATA CHO CÁC VIDEO THỰC TẾ
# ==========================================
import pandas as pd

META_XLSX = REPO_DIR / "driver_behavior_dataset_v01" / "annotations" / "metadata.xlsx"
META_CSV = REPO_DIR / "driver_behavior_dataset_v01" / "annotations" / "metadata.csv"

df_meta = pd.read_excel(META_XLSX)
print(f"Metadata gốc: {len(df_meta)} rows ({df_meta['source_video'].nunique()} videos)")

raw_root = REPO_DIR / "driver_behavior_dataset_v01" / "raw"
existing_mask = df_meta["source_video"].apply(lambda v: (raw_root / str(v).strip()).exists())
df_available = df_meta[existing_mask].copy()

if len(df_available) == 0:
    print("⚠️ CẢNH BÁO: Tên 2 file video upload chưa khớp với cột 'source_video' trong metadata.xlsx!")
    print("Các video trong metadata gốc:", df_meta['source_video'].unique().tolist())
    print("Các video thực tế trong raw/public:", [v.name for v in available_videos])
else:
    print(f"Số dòng metadata tương ứng với các video thực tế: {len(df_available)} rows")
    print(f"Source videos: {df_available['source_video'].unique().tolist()}")
    print(f"Subjects: {df_available['subject_id'].unique().tolist()}")
    print("\nPhân bố nhãn hành vi:")
    print(df_available["label"].value_counts())
    
    # Ghi đè metadata.xlsx và xuất metadata.csv
    df_available.to_excel(META_XLSX, index=False)
    df_available.to_csv(META_CSV, index=False)
    print(f"\nĐã cập nhật: {META_XLSX.name} và {META_CSV.name}")


## 6. Kiểm Kê & Xác Thực Metadata (Pipeline Step 1)
Chạy `inventory.py` để quét thông số video (fps, duration, resolution) và `validate_metadata.py` để kiểm tra toàn vẹn dữ liệu.


In [ ]:
# ==========================================
# 6. INVENTORY & VALIDATE METADATA
# ==========================================
print("=== CHẠY INVENTORY.PY ===")
subprocess.run([sys.executable, "driver_behavior_dataset_v01/scripts/inventory.py", "--overwrite", "--verbose"], check=True)

print("\n=== CHẠY VALIDATE_METADATA.PY ===")
subprocess.run([sys.executable, "driver_behavior_dataset_v01/scripts/validate_metadata.py", "--verbose"], check=True)


## 7. Cắt Video Thành Các Segment Hành Vi (Pipeline Step 2)
Dùng FFmpeg cắt các đoạn hành vi theo start/end timestamp trong metadata và lưu vào `segments/<label>/`.


In [ ]:
# ==========================================
# 7. CẮT VIDEO SEGMENTS (FFMPEG)
# ==========================================
print("=== CẮT VIDEO SEGMENTS ===")
subprocess.run([
    sys.executable, "driver_behavior_dataset_v01/scripts/cut_segments.py",
    "--workers", "2",
    "--overwrite",
    "--verbose"
], check=True)

segments_dir = REPO_DIR / "driver_behavior_dataset_v01" / "segments"
print("\nThống kê số lượng video clip segment đã cắt:")
for folder in sorted(segments_dir.iterdir()):
    if folder.is_dir():
        clips = list(folder.glob("*.mp4"))
        print(f"  - Lớp {folder.name:12s}: {len(clips):3d} video clips")


## 8. Chia Tập Train / Val / Test (Pipeline Step 3)
Phân chia độc lập theo Subject (chống Data Leakage). 
- Nếu có từ 2 subject trở lên: chia theo subject độc lập (1 subject cho train, 1 subject cho val).
- Nếu cả 2 video thuộc cùng 1 subject: chia theo video để đảm bảo cả `train` và `val` đều có dữ liệu và không rò rỉ khung hình giữa các video.


In [ ]:
# ==========================================
# 8. TẠO SPLITS TRAIN / VAL / TEST
# ==========================================
df_curr_meta = pd.read_csv(META_CSV)
valid_meta = df_curr_meta[df_curr_meta["quality"].str.lower() != "bad"]
unique_subs = valid_meta["subject_id"].unique()

SPLITS_CSV = REPO_DIR / "driver_behavior_dataset_v01" / "annotations" / "splits.csv"

if len(unique_subs) >= 2:
    print(f"Có {len(unique_subs)} subject ({unique_subs.tolist()}). Chia tập theo subject:")
    subprocess.run([
        sys.executable, "driver_behavior_dataset_v01/scripts/make_splits.py",
        "--train-ratio", "0.6",
        "--val-ratio", "0.4",
        "--test-ratio", "0.0",
        "--overwrite"
    ], check=True)
else:
    print(f"Chỉ có 1 subject ({unique_subs[0]}). Phân bổ theo từng video vào Train và Val:")
    videos = valid_meta["source_video"].unique()
    train_vid = videos[0]
    val_vid = videos[1] if len(videos) > 1 else videos[0]
    
    valid_meta["original_subject_id"] = valid_meta["subject_id"]
    valid_meta["subject_id"] = valid_meta["source_video"].apply(
        lambda v: f"{unique_subs[0]}_A" if v == train_vid else f"{unique_subs[0]}_B"
    )
    valid_meta.to_csv(META_CSV, index=False)
    
    splits_df = pd.DataFrame([
        {"subject_id": f"{unique_subs[0]}_A", "split": "train"},
        {"subject_id": f"{unique_subs[0]}_B", "split": "val"}
    ])
    splits_df.to_csv(SPLITS_CSV, index=False)

splits_df = pd.read_csv(SPLITS_CSV)
print("\nKết quả phân chia (splits.csv):")
print(splits_df)


## 9. Trích Xuất MediaPipe Face Landmarks (Pipeline Step 4)
Chạy MediaPipe Face Landmarker trích xuất 478 tọa độ khuôn mặt 3D theo thời gian thực và lưu thành các file `.npz` đặc trưng hình học.


In [ ]:
# ==========================================
# 9. TRÍCH XUẤT MEDIA PIPE LANDMARKS (.npz)
# ==========================================
print("=== TRÍCH XUẤT ĐẶC TRƯNG LANDMARKS ===")
subprocess.run([
    sys.executable, "driver_behavior_dataset_v01/scripts/extract_landmarks.py",
    "--model-path", str(LANDMARKER_PATH),
    "--workers", "2",
    "--overwrite",
    "--verbose"
], check=True)

landmarks_dir = REPO_DIR / "driver_behavior_dataset_v01" / "features" / "landmarks"
npz_files = list(landmarks_dir.glob("*.npz"))
print(f"\nĐã trích xuất thành công {len(npz_files)} file .npz vào {landmarks_dir}")

import numpy as np
if npz_files:
    sample = np.load(npz_files[0])
    print(f"\nKiểm tra cấu trúc file mẫu ({npz_files[0].name}):")
    for k in sample.files:
        print(f"  - {k:20s}: shape={sample[k].shape}, dtype={sample[k].dtype}")


## 10. Trích Xuất Frames (Dành Cho CNN-LSTM nếu cần)
Trích xuất khung hình JPG và lưu đóng gói file `.npy` tối ưu I/O cho mô hình CNN-LSTM.


In [ ]:
# ==========================================
# 10. TRÍCH XUẤT FRAMES CHO CNN-LSTM (TÙY CHỌN)
# ==========================================
print("=== TRÍCH XUẤT FRAMES ===")
subprocess.run([
    sys.executable, "driver_behavior_dataset_v01/scripts/extract_frames.py",
    "--pack",
    "--fps", "10",
    "--overwrite"
], check=True)

frames_dir = REPO_DIR / "driver_behavior_dataset_v01" / "processed" / "frames"
print(f"Đã trích xuất frames vào {frames_dir}")


## 11. Huấn Luyện Mô Hình (Landmark-LSTM)
Huấn luyện mô hình chuỗi thời gian LSTM trên đặc trưng landmark với trọng số cân bằng lớp và early stopping.


In [ ]:
# ==========================================
# 11. HUẤN LUYỆN MODEL LANDMARK-LSTM
# ==========================================
CONFIG_PATH = "configs/models/landmark_lstm.yaml"
RUN_NAME = "landmark_lstm_run"

print(f"Bắt đầu huấn luyện mô hình với cấu hình: {CONFIG_PATH}...")
subprocess.run([
    sys.executable, "-m", "dbm.train",
    "--config", CONFIG_PATH,
    "--run-name", RUN_NAME,
    "--override", "batch_size=16",
    "--override", "max_epochs=25",
    "--override", "lr=0.001"
], check=True)


## 12. Đánh Giá & Vẽ Biểu Đồ (Loss Curves & Confusion Matrix)


In [ ]:
# ==========================================
# 12. ĐÁNH GIÁ VÀ HIỂN THỊ ĐỒ THỊ
# ==========================================
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = REPO_DIR / "outputs" / RUN_NAME
best_ckpt = run_dir / "best.pt"
print(f"Đánh giá checkpoint tốt nhất: {best_ckpt}")

# Đánh giá lại trên tập val
subprocess.run([
    sys.executable, "-m", "dbm.evaluate",
    "--checkpoint", str(best_ckpt),
    "--split", "val"
], check=True)

# Hiển thị biểu đồ huấn luyện và ma trận nhầm lẫn
fig, axs = plt.subplots(1, 2, figsize=(16, 6))

lc_img_path = run_dir / "learning_curves.png"
if lc_img_path.exists():
    img1 = mpimg.imread(str(lc_img_path))
    axs[0].imshow(img1)
    axs[0].set_title("Biểu Đồ Học Tập (Train/Val Loss & Macro-F1)", fontsize=14)
    axs[0].axis("off")

cm_img_path = run_dir / "confusion_matrix.png"
if cm_img_path.exists():
    img2 = mpimg.imread(str(cm_img_path))
    axs[1].imshow(img2)
    axs[1].set_title("Ma Trận Nhầm Lẫn (Validation Confusion Matrix)", fontsize=14)
    axs[1].axis("off")

plt.tight_layout()
plt.show()


## 13. Thử Nghiệm Dự Đoán Video (Sliding Window Inference)
Chạy thử nghiệm suy luận trượt (Sliding Window) qua một đoạn clip bất kỳ bằng `dbm.infer_video`.


In [ ]:
# ==========================================
# 13. CHẠY SUY LUẬN TRƯỢT TRÊN VIDEO
# ==========================================
all_segments = list(segments_dir.glob("*/*.mp4"))
if all_segments:
    sample_clip = all_segments[0]
    output_timeline = run_dir / "timeline_pred.csv"
    print(f"Thử nghiệm suy luận trên video clip: {sample_clip.name} (Lớp thực: {sample_clip.parent.name})")
    
    subprocess.run([
        sys.executable, "-m", "dbm.infer_video",
        "--video", str(sample_clip),
        "--checkpoint", str(best_ckpt),
        "--output", str(output_timeline),
        "--smoothing", "ema"
    ], check=True)
    
    if output_timeline.exists():
        df_pred = pd.read_csv(output_timeline)
        print("\nBảng kết quả dự đoán từng cửa sổ:")
        print(df_pred.head(10))


## 14. Nén Toàn Bộ Kết Quả & Checkpoint Để Tải Về


In [ ]:
# ==========================================
# 14. NÉN KẾT QUẢ VÀ CHECKPOINT
# ==========================================
zip_out = WORK_DIR / "driver_behaviour_results.zip"
print(f"Đang nén outputs vào {zip_out}...")
subprocess.run(["zip", "-r", str(zip_out), "outputs/"], check=True)
print(f" Hoàn tất! Bạn có thể tải file '{zip_out.name}' trực tiếp tại panel Output bên phải của Kaggle.")
